# EXP05 — Auditoria de transações e eventos

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. Dados necessários

Forneça `data/processed/exp05_transacoes_auditadas.csv`, contendo pelo menos `contrato`, `tx_hash`, `fonte`, `conferido`. Uma linha representa **uma transação conferida**; o campo `conferido` deve ser preenchido por quem revisou uma fonte independente, não inferido pelo programa.

In [ ]:
import pandas as pd
from ethmix.execucao import abrir_dados

caminho = RAIZ / "data" / "processed" / "exp05_transacoes_auditadas.csv"
dados = abrir_dados(caminho, ["contrato", "tx_hash", "fonte", "conferido"])
print("Linhas:", len(dados), "| Contratos:", dados.contrato.nunique())
print(dados.groupby("contrato").size().rename("amostras"))
display(dados.head())

## 3. Checar a amostra

O relatório histórico registra 18/18 transações amostradas conferidas e 586 logs relacionados às 18 transações. Isso **não** significa que todas as operações de cada contrato foram auditadas.

In [ ]:
valores = dados["conferido"].astype(str).str.lower().str.strip()
confirmadas = valores.isin(["true", "1", "sim", "yes"]).sum()
print("Transações marcadas como conferidas:", confirmadas, "/", len(dados))
print("O campo informa a avaliação humana registrada; não faz consulta externa.")